In [1]:
from tqdm import tqdm
import json
import re
import requests
import time

file_path = '../../data/reports/zeroshot/e709f8b3-bf04-477a-80ef-6e371bb04663.json'

with open(file_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

In [2]:
wikidata_uris = set()
dbpedia_uris = set()
other_uris = set()

# Regex para encontrar URIs que começam com http ou <http
uri_pattern = r"http.*?(?=http|\s|>|\}\)|\)\}|\}|\)\)\)|\)\)|\;|\)\])"

# Itera sobre o JSON
for gold_query in json_data['data']:
    for generated_query in gold_query['Generated Queries']:
        match = re.findall(uri_pattern, generated_query['SparQL Query'])
        for m in match:
            
            if m.endswith('.') or m.endswith('/"') or m.endswith('|') or m.endswith('*'):
                m = m[:-1]
            if (m.endswith(')') and ('(' not in m)):
                m = m[:-1]

            if "wikidata" in m:
                wikidata_uris.add(m)
            elif "dbpedia" in m:
                dbpedia_uris.add(m)
            else:
                other_uris.add(m)

In [3]:
def is_fake_wikidata_uri(uri):

    time.sleep(2)
    
    url = "https://query.wikidata.org/sparql"

    headers = {
        "Accept": "application/json"
    }

    query = """
    ASK WHERE {
        FILTER NOT EXISTS{
            {
                <?uri> ?p ?o
            } UNION {
                ?s <?uri> ?o
            } UNION {
                ?s ?p <?uri>
            }
        }
    }
    """

    return requests.get(url, headers=headers, params={'query': query.replace('?uri',uri)})

def is_fake_dbpedia_uri(uri):

    time.sleep(2)
    
    url = "https://dbpedia.org/sparql"

    headers = {
        "Accept": "application/json"
    }

    query = """
    ASK WHERE {
        FILTER NOT EXISTS{
            {
                <?uri> ?p ?o
            } UNION {
                ?s <?uri> ?o
            } UNION {
                ?s ?p <?uri>
            }
        }
    }
    """

    return requests.get(url, headers=headers, params={'query': query.replace('?uri',uri)})

In [ ]:
report = []

for uri in tqdm(wikidata_uris):

    result = {"uri": uri}

    is_fake = is_fake_wikidata_uri(uri)
    result["status_code"] = is_fake.status_code

    try:
        if   is_fake.json()['boolean'] ==  True: result["is_fake"] = True
        elif is_fake.json()['boolean'] == False: result["is_fake"] = False
    except Exception as e:
        result["is_fake"] = None
        result["error"] = str(e)

    report.append(result)

with open("wikidata_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

In [ ]:
report = []

for uri in tqdm(dbpedia_uris):

    result = {"uri": uri}

    is_fake = is_fake_dbpedia_uri(uri)
    result["status_code"] = is_fake.status_code

    try:
        if   is_fake.json()['boolean'] ==  True: result["is_fake"] = True
        elif is_fake.json()['boolean'] == False: result["is_fake"] = False
    except Exception as e:
        result["is_fake"] = None
        result["error"] = str(e)

    report.append(result)

with open("dbpedia_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

In [ ]:
dbpedia_report_path  = 'dbpedia_report.json'
wikidata_report_path = 'wikidata_report.json'

with open(dbpedia_report_path, 'r', encoding='utf-8') as f:
    dbpedia_report_data = json.load(f)

with open(wikidata_report_path, 'r', encoding='utf-8') as f:
    wikidata_report_data = json.load(f)

In [ ]:
total_uris = 0
fake_count = 0
not_fake_count = 0
error_count = 0

for item in dbpedia_report_data:
    total_uris += 1
    try:
        if item.get('status_code') != 200:
            error_count += 1
            continue
        is_fake = item['is_fake']
        if is_fake:
            fake_count += 1
        else:
            not_fake_count += 1
    except Exception:
        error_count += 1

print("================== DBPEDIA REPORT ==================")
print(f"--------Total analysed URIs: {total_uris} - {100}%")
print(f"------------------Fake URIs: {fake_count} - {(fake_count/total_uris)*100}%")
print(f"------------------True URIs: {not_fake_count} - {(not_fake_count/total_uris)*100}%")
print(f"URIs with processing errors:  {error_count}  - {(error_count/total_uris)*100}%")


In [ ]:
total_uris = 0
fake_count = 0
not_fake_count = 0
error_count = 0

for item in wikidata_report_data:
    total_uris += 1
    try:
        if item.get('status_code') != 200:
            error_count += 1
            continue
        is_fake = item['is_fake']
        if is_fake:
            fake_count += 1
        else:
            not_fake_count += 1
    except Exception:
        error_count += 1

print("================== WIKIDATA REPORT ==================")
print(f"--------Total analysed URIs: {total_uris} - {100}%")
print(f"------------------Fake URIs: {fake_count} - {(fake_count/total_uris)*100}%")
print(f"------------------True URIs: {not_fake_count} - {(not_fake_count/total_uris)*100}%")
print(f"URIs with processing errors:  {error_count}  - {(error_count/total_uris)*100}%")